In [ ]:
# =========================
# CELL 1: setup + install
# =========================

import os
import random
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/moses_full_reproduction")
REPO_DIR = Path("/content/MoSEs")
REPO_CACHE = DRIVE_ROOT / "repo_cache"
HF_CACHE = DRIVE_ROOT / "hf_cache"
RESULTS_ROOT = DRIVE_ROOT / "final_outputs"

for p in [DRIVE_ROOT, REPO_CACHE, HF_CACHE, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def run_cmd(cmd, cwd=None, env=None, shell=False, check=True):
    printable = (
        cmd if isinstance(cmd, str) else " ".join(shlex.quote(str(x)) for x in cmd)
    )
    print("\n" + "=" * 100)
    print(printable)
    print("=" * 100)
    result = subprocess.run(
        cmd, cwd=cwd, env=env, shell=shell, capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode,
            cmd,
            output=result.stdout,
            stderr=result.stderr,
        )
    return result


def copytree_merge(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        s = item
        d = dst / item.name
        if s.is_dir():
            copytree_merge(s, d)
        else:
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(s, d)


# fresh clone
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", "https://github.com/HuskyDevClub/MoSEs.git", str(REPO_DIR)])

# restore cached outputs if they exist
for rel in ["weights", "logs"]:
    copytree_merge(REPO_CACHE / rel, REPO_DIR / rel)

# restore previously generated split_datasets* folders
cached_data_dir = REPO_CACHE / "data"
if cached_data_dir.exists():
    for p in cached_data_dir.glob("split_datasets*"):
        copytree_merge(p, REPO_DIR / "data" / p.name)

# install dependencies
run_cmd(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

run_cmd(["nvidia-smi"], check=False)
print("Setup complete.")

In [ ]:
# =========================
# CELL 2: configuration + helpers
# =========================

import json
import re
import pandas as pd
from pathlib import Path

DATA_INPUT_DIR = REPO_DIR / "data" / "doc4split"

MAIN_KEYS = ["cmv", "sci", "wp", "xsum"]
LOW_KEYS = ["cnn", "dialogsum", "imdb", "pubmedqa"]

# You can flip any of these to False if you need to rerun only part of the pipeline.
RUN_FAST = True
RUN_LASTDE = True
RUN_ROBERTA = True

# Official example scripts clearly show main fast/lastde use *_train as reference set.
# The released roberta example points to the unsplit folder; keep this toggle if you want to match that behavior.
ROBERTA_MAIN_USE_FULL_FOLDER = True

DETECTORS = {
    "fast": {
        "enabled": RUN_FAST,
        "crit_type": "fast",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_fast",
        "extra_args": [],
    },
    "lastde": {
        "enabled": RUN_LASTDE,
        "crit_type": "lastde",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_lastde",
        "extra_args": ["--embed_size", "4", "--epsilon", "8", "--tau_prime", "15"],
    },
    "roberta_base": {
        "enabled": RUN_ROBERTA,
        "crit_type": "roberta",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_roberta_base",
        "extra_args": ["--roberta_model_name", "roberta-base-openai-detector"],
    },
}


def sync_outputs_to_drive():
    (REPO_CACHE / "data").mkdir(parents=True, exist_ok=True)
    for p in (REPO_DIR / "data").glob("split_datasets*"):
        copytree_merge(p, REPO_CACHE / "data" / p.name)
    copytree_merge(REPO_DIR / "weights", REPO_CACHE / "weights")
    copytree_merge(REPO_DIR / "logs", REPO_CACHE / "logs")


def dataset_key_from_json_name(name):
    key = Path(name).stem.lower()
    if key.endswith("_dataset"):
        key = key[:-8]
    return key


def subset_json_folder(src_dir, dst_dir, allowed_keys):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_dir.glob("*.json"):
        if dataset_key_from_json_name(p.name) in allowed_keys:
            shutil.copy2(p, dst_dir / p.name)


def split_json_folder_dataset_aware(
    src_dir, train_dir, test_dir, main_test_per_label=100, seed=42
):
    """Split each JSON file in src_dir into train/test sets.

    Paper specifies 1800 ref / 200 test for main datasets (100 per label)
    and 200 ref / 200 test for low-resource datasets (half per label).
    """
    src_dir = Path(src_dir)
    train_dir = Path(train_dir)
    test_dir = Path(test_dir)
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    for p in sorted(src_dir.glob("*.json")):
        key = dataset_key_from_json_name(p.name)
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)

        label0 = [x for x in data if int(x["label"]) == 0]
        label1 = [x for x in data if int(x["label"]) == 1]
        rng.shuffle(label0)
        rng.shuffle(label1)

        if key in LOW_KEYS:
            # Paper: 200 ref / 200 test  ->  half per label to test, half to reference
            test_n = min(len(label0), len(label1)) // 2
        else:
            # Paper: 1800 ref / 200 test  ->  100 per label for test
            test_n = min(main_test_per_label, len(label0) // 2, len(label1) // 2)

        test_data = label0[:test_n] + label1[:test_n]
        train_data = label0[test_n:] + label1[test_n:]

        with open(test_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(test_data, f, ensure_ascii=False, indent=2)
        with open(train_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(train_data, f, ensure_ascii=False, indent=2)

        print(
            f"{p.name}: train={len(train_data)}, test={len(test_data)}, test_per_label={test_n}"
        )


def prepare_detector_views(processed_dir):
    """Split the full 8-dataset processed folder into train/test, then create
    main-only and low-only test subsets for evaluation iteration.

    Matches the paper: SAR and CTE both operate on the full 8-class SRR,
    while evaluation iterates over 4 main or 4 low-resource test files.
    """
    processed_dir = Path(processed_dir)

    # Full train/test split (all 8 datasets)
    full_train = Path(str(processed_dir) + "_train")
    full_test = Path(str(processed_dir) + "_test")

    # Test subsets for evaluation iteration only
    main_test = Path(str(processed_dir) + "_main_test")
    low_test = Path(str(processed_dir) + "_low_test")

    # Split full folder into train/test (100 per label for main, half for low)
    if (
        not full_train.exists()
        or not list(full_train.glob("*.json"))
        or not full_test.exists()
        or not list(full_test.glob("*.json"))
    ):
        split_json_folder_dataset_aware(
            processed_dir, full_train, full_test, main_test_per_label=100, seed=42
        )

    # Subset the test folder into main / low for CTE evaluation iteration
    if not main_test.exists() or not list(main_test.glob("*.json")):
        subset_json_folder(full_test, main_test, MAIN_KEYS)
    if not low_test.exists() or not list(low_test.glob("*.json")):
        subset_json_folder(full_test, low_test, LOW_KEYS)

    return {
        "full": processed_dir,  # unsplit, all 8 datasets
        "train": full_train,  # reference set, all 8 datasets
        "test": full_test,  # test set, all 8 datasets
        "main_test": main_test,  # 4 main dataset test files
        "low_test": low_test,  # 4 low-resource dataset test files
    }


def find_best_sar_weights(output_dir):
    """Find the SAR weight file with the highest epoch number (= best val accuracy).

    SAR.py saves a new file each time val_acc improves, so the highest
    epoch number corresponds to the best model.
    """
    output_dir = Path(output_dir)
    pts = list(output_dir.glob("subcentroids_head_epoch*.pt"))
    if not pts:
        return None

    def epoch_num(p):
        m = re.search(r"epoch(\d+)", p.name)
        return int(m.group(1)) if m else 0

    return max(pts, key=epoch_num)


def flatten_json(x, prefix=""):
    out = {}
    if isinstance(x, dict):
        for k, v in x.items():
            out.update(flatten_json(v, f"{prefix}{k}." if prefix else f"{k}."))
    elif isinstance(x, list):
        if len(x) <= 20:
            out[prefix[:-1]] = str(x)
    else:
        out[prefix[:-1]] = x
    return out


print("Helpers ready.")

In [ ]:
# =========================
# CELL 3: verify input data + build processed folders for all detectors
# =========================

csvs = sorted(DATA_INPUT_DIR.glob("*.csv"))
print("Found CSV files:")
for p in csvs:
    print(" -", p.name)

assert len(csvs) > 0, f"No CSV files found in {DATA_INPUT_DIR}"


def build_processed_folder(det_name, cfg):
    out_dir = REPO_DIR / "data" / cfg["out_name"]
    jsons = list(out_dir.glob("*.json")) if out_dir.exists() else []
    if len(jsons) >= 8:
        print(f"[SKIP] {det_name}: found {len(jsons)} processed JSONs in {out_dir}")
        return out_dir

    cmd = [
        sys.executable,
        "split_datasets.py",
        "--input_directory",
        str(DATA_INPUT_DIR),
        "--output_directory",
        str(out_dir),
        "--embedding_type",
        "encode",
        "--embedding_model_name",
        "BAAI/bge-m3",
        "--scoring_model_name",
        "EleutherAI/gpt-neo-2.7B",
        "--batch_size",
        "500",
        "--crit_type",
        cfg["crit_type"],
        "--cache_dir",
        str(HF_CACHE),
    ] + cfg["extra_args"]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()
    return out_dir


PROCESSED_DIRS = {}

for det_name, cfg in DETECTORS.items():
    if not cfg["enabled"]:
        continue
    PROCESSED_DIRS[det_name] = build_processed_folder(det_name, cfg)

print("\nProcessed dirs:")
for k, v in PROCESSED_DIRS.items():
    print(k, "->", v)

In [ ]:
# =========================
# CELL 4: create train/test splits and test subsets for all detectors
# =========================

VIEW_PATHS = {}

for det_name, processed_dir in PROCESSED_DIRS.items():
    print(f"\nPreparing views for {det_name}")
    VIEW_PATHS[det_name] = prepare_detector_views(processed_dir)

sync_outputs_to_drive()

for det_name, views in VIEW_PATHS.items():
    print(f"\n=== {det_name} ===")
    for k, v in views.items():
        count = len(list(Path(v).glob("*.json"))) if Path(v).exists() else 0
        print(f"{k:>12}: {v} ({count} json files)")

In [ ]:
# =========================
# CELL 5: train SAR on the full 8-class SRR (matching the paper)
# =========================

# Paper Section 3.3: SAR operates on the full SRR containing all 8 stylistic categories.
# The original run_train.sh trains SAR on the complete (unsplit) fast-processed folder.
# We train on the _train split (all 8 datasets) so test data is excluded.

SAR_OUT = REPO_DIR / "weights" / "model_output_c16"


def train_sar_if_needed(dataset_folder, output_dir):
    class_path = output_dir / "class_names.json"
    weight_path = find_best_sar_weights(output_dir)

    if weight_path is not None and class_path.exists():
        print(f"[SKIP] SAR already exists at {output_dir} (best: {weight_path.name})")
        return weight_path, class_path

    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "SAR.py",
        "train",
        "--model_name",
        "BAAI/bge-m3",
        "--datasets_folder",
        str(dataset_folder),
        "--embedding_type",
        "encode",
        "--num_epochs",
        "100",
        "--batch_size",
        "32",
        "--num_subcentroids",
        "4",
        "--output_dir",
        str(output_dir),
    ]

    # If a checkpoint exists from a previous interrupted run, resume from it
    checkpoint_path = output_dir / "checkpoint_latest.pt"
    if checkpoint_path.exists():
        print(f"[RESUME] Found checkpoint at {checkpoint_path}, resuming training...")
        cmd += ["--resume_checkpoint", str(checkpoint_path)]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()

    weight_path = find_best_sar_weights(output_dir)
    assert (
        weight_path is not None
    ), f"No SAR weights found in {output_dir} after training"
    return weight_path, class_path


assert (
    "fast" in VIEW_PATHS
), "fast detector processed folder is needed to train SAR as in the released examples."

# Train a single SAR on the full _train folder (all 8 datasets)
SAR_PATH, SAR_CLASS = train_sar_if_needed(VIEW_PATHS["fast"]["train"], SAR_OUT)

print("SAR weights:", SAR_PATH)
print("SAR classes:", SAR_CLASS)

In [ ]:
# =========================
# CELL 6: run CTE evaluation for all detectors on main + low-resource
# =========================

# Paper: CTE uses the full SRR (all 8 datasets) as the reference pool.
# Original run_cte_fast_detect.sh uses the _train folder (all 8 datasets).
# Original run_cte_roberta_base.sh uses the unsplit folder (all 8 datasets).

from concurrent.futures import ThreadPoolExecutor, as_completed

DONE_DIR = REPO_DIR / "logs" / "done_markers"
DONE_DIR.mkdir(parents=True, exist_ok=True)

# Max parallel CTE subprocesses. Each CTE process uses ~3-4GB VRAM (BGE-M3 + SAR).
# A100 40GB -> 4 workers safe, A100 80GB -> 4+ workers safe.
MAX_CTE_WORKERS = 4


def run_cte_single(
    test_file, result_prefix, datasets_folder, sar_path, class_path, marker_prefix
):
    """Run CTE.py on a single test file. Returns (marker_name, success, error)."""
    test_file = Path(test_file)
    result_prefix = Path(result_prefix)
    result_prefix.parent.mkdir(parents=True, exist_ok=True)
    marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"

    if marker.exists():
        return marker.name, True, "skipped"

    cmd = [
        sys.executable,
        "CTE.py",
        "--file_path",
        str(test_file),
        "--result_file_path",
        str(result_prefix),
        "--datasets_folder",
        str(datasets_folder),
        "--sar_path",
        str(sar_path),
        "--class_path",
        str(class_path),
    ]
    try:
        result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        if result.returncode != 0:
            return marker.name, False, result.stderr
        marker.touch()
        return marker.name, True, result.stdout[-200:] if result.stdout else ""
    except Exception as e:
        return marker.name, False, str(e)


# Collect all CTE jobs
cte_jobs = []

for det_name, views in VIEW_PATHS.items():
    if det_name == "roberta_base" and ROBERTA_MAIN_USE_FULL_FOLDER:
        ref_folder = views["full"]
    else:
        ref_folder = views["train"]

    # Main test files
    main_test_dir = Path(views["main_test"])
    for test_file in sorted(main_test_dir.glob("*.json")):
        cte_jobs.append(
            {
                "test_file": test_file,
                "result_prefix": REPO_DIR / "logs" / "reproduction" / "main" / det_name,
                "datasets_folder": ref_folder,
                "sar_path": SAR_PATH,
                "class_path": SAR_CLASS,
                "marker_prefix": f"main__{det_name}",
                "label": f"{det_name}/main/{test_file.stem}",
            }
        )

    # Low-resource test files
    low_test_dir = Path(views["low_test"])
    for test_file in sorted(low_test_dir.glob("*.json")):
        cte_jobs.append(
            {
                "test_file": test_file,
                "result_prefix": REPO_DIR / "logs" / "reproduction" / "low" / det_name,
                "datasets_folder": views["train"],
                "sar_path": SAR_PATH,
                "class_path": SAR_CLASS,
                "marker_prefix": f"low__{det_name}",
                "label": f"{det_name}/low/{test_file.stem}",
            }
        )

print(f"Total CTE jobs: {len(cte_jobs)}, max parallel workers: {MAX_CTE_WORKERS}")

# Run jobs in parallel
completed = 0
failed = []

with ThreadPoolExecutor(max_workers=MAX_CTE_WORKERS) as executor:
    future_to_label = {}
    for job in cte_jobs:
        future = executor.submit(
            run_cte_single,
            job["test_file"],
            job["result_prefix"],
            job["datasets_folder"],
            job["sar_path"],
            job["class_path"],
            job["marker_prefix"],
        )
        future_to_label[future] = job["label"]

    for future in as_completed(future_to_label):
        label = future_to_label[future]
        marker_name, success, msg = future.result()
        completed += 1
        status = "OK" if success else "FAIL"
        print(f"[{completed}/{len(cte_jobs)}] {status}: {label} ({msg[:80]})")
        if not success:
            failed.append((label, msg))

sync_outputs_to_drive()

if failed:
    print(f"\n{len(failed)} jobs FAILED:")
    for label, msg in failed:
        print(f"  {label}: {msg[:200]}")
else:
    print(f"\nAll {len(cte_jobs)} CTE jobs finished successfully.")

In [ ]:
# =========================
# CELL 7: collect outputs into a summary table + save to Drive
# =========================

records = []

for p in sorted((REPO_DIR / "logs").rglob("*.json")):
    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        flat = flatten_json(obj)
        rec = {"file": str(p.relative_to(REPO_DIR))}
        for k, v in flat.items():
            if isinstance(v, (int, float, str)):
                rec[k] = v
        records.append(rec)
    except Exception as e:
        records.append({"file": str(p.relative_to(REPO_DIR)), "parse_error": str(e)})

df = pd.DataFrame(records).fillna("")
summary_csv = RESULTS_ROOT / "reproduction_log_summary.csv"
df.to_csv(summary_csv, index=False)

print(f"Saved summary to: {summary_csv}")
display(df.head(50))

# also make a zip of logs and weights
logs_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_logs"), "zip", REPO_DIR / "logs"
)
weights_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_weights"), "zip", REPO_DIR / "weights"
)

print("Logs zip   :", logs_zip)
print("Weights zip:", weights_zip)

sync_outputs_to_drive()
print("Everything synced back to Drive.")